### RaschPy RSM worked example

This notebook works through a sample Rasch analysis of a simulated data set (1,000 persons, 8 items with a maximum score of 5, no missing data), taking you through the relevant commands step by step, with notes before each cell. Relevant outputs will appear below each cell.

Import the modules and set the working directory (here called `my_working_directory`) to where you want to save your output files.

In [ ]:
import raschpy as rp
import os

os.chdir('my_working_directory')

**A note on data validation**

Every model constructor validates the item-response network automatically at instantiation (`validate=True` by default) and warns if it's disconnected — if there's no chain of persons and items linking every item to every other, the resulting item locations aren't on a common scale, even though `calibrate()` will still run without error. This is worth seeing happen once. Here we build a small, deliberately disconnected data set: two groups of persons who each only answer a disjoint set of items, with no item shared between the groups to link them:

In [ ]:
sim_disconnected = rp.RSM_Sim(no_of_items=8, no_of_persons=80, max_score=4, seed=99)
responses_disconnected = sim_disconnected.responses.copy()
responses_disconnected.iloc[:40, 4:] = float('nan')   # first 40 persons: only answer items 1-4
responses_disconnected.iloc[40:, :4] = float('nan')    # last 40 persons: only answer items 5-8

broken_rsm = rp.RSM(responses_disconnected, max_score=4)   # raises a UserWarning

`connectivity_status` records the diagnosis, including which items ended up in which isolated sub-group:

In [ ]:
broken_rsm.connectivity_status

The rest of this notebook uses a single, fully-connected simulated data set, so this warning won't come up again.

Simulate a data set. Passing `seed=42` makes the simulation fully reproducible — rerunning this notebook will always generate the same data set. 12 items, 1,000 persons, no missing data.

In [ ]:
sim = rp.RSM_Sim(no_of_items=12, no_of_persons=1000, max_score=5, missing=0, seed=42)
sim.responses.to_csv('rsm_scores.csv')

If you have your own response data saved to a CSV file instead of simulating it, use `loadup_rsm()` to load and validate it. Demonstrated here by reloading the file we just saved:

In [ ]:
data, invalid_responses = rp.loadup_rsm('rsm_scores.csv', max_score=5)

Check the data - view the first two lines

In [ ]:
data.head(2)

Check for any invalid responses (not usable for estimation purposes and excluded)

In [ ]:
invalid_responses

Create an RSM object. Passing the simulation object `sim` directly (rather than the reloaded `data`) attaches the generating parameters under `rsm.generating`, which lets us check parameter recovery further down, and also means `max_score` doesn't need to be supplied again. If you're analysing your own data, pass a DataFrame and `max_score` instead (e.g. `rp.RSM(data, max_score)`).

In [ ]:
rsm = rp.RSM(sim)

Generate item estimates. The `%%time` "magic function" returns the time taken to run the cell contents (algorithm run time in this case).

In [ ]:
%%time
rsm.calibrate()

Check the item location estimates

In [ ]:
rsm.items

Since this is simulated data, we know the true generating item locations (`slm.generating.items`) and can check how closely the calibration recovered them. The helper below plots generating vs. estimated values, with an identity line (dashed dark red) and a fitted regression line (dashed red) — the closer the points hug the identity line, the better the recovery. Also displays the Pearson correlation, SD ratio, regression coefficient and RMSE:

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

def recovery_plot(generating, estimated, label, filename):
    x, y = np.asarray(generating), np.asarray(estimated)
    fig, ax = plt.subplots()
    ax.scatter(x, y, alpha=0.6)
    lo, hi = min(x.min(), y.min()), max(x.max(), y.max())
    ax.plot([lo, hi], [lo, hi], color='DarkRed', label='Identity')
    m, b = np.polyfit(x, y, 1)
    ax.plot([lo, hi], [m * lo + b, m * hi + b], color='Red', linestyle='--', label='Regression')
    ax.set_xlabel(f'Generating {label}')
    ax.set_ylabel(f'Estimated {label}')
    ax.set_aspect('equal')
    ax.legend()
    plt.savefig(filename)
    plt.show()
    print(f'{label} Pearson correlation:              {round(np.corrcoef(x, y)[0, 1], 3)}')
    print(f'{label} SD ratio (estimated / generating): {round(y.std() / x.std(), 3)}')
    print(f'{label} regression coefficient:            {round(m, 3)}')
    print(f'{label} RMSE:                              {round(np.sqrt(((x - y) ** 2).mean()), 3)}')

recovery_plot(sim.items, rsm.items, 'item location', 'my_rsm_item_recovery.png')

Generate a table of item statistics (and check run time), and save to file

In [ ]:
%%time
rsm.item_stats_df(full=True)
rsm.item_stats.to_csv('rsm_item_stats.csv')

Check the item statistics table

In [ ]:
rsm.item_stats

Generate a table of threshold statistics (and check run time), and save to file

In [ ]:
%%time
rsm.threshold_stats_df(full=True)
rsm.threshold_stats.to_csv('rsm_threshold_stats.csv')

Check the threshold statistics table

In [ ]:
rsm.threshold_stats

And the same recovery check for the shared threshold vector, against `sim.thresholds`:

In [ ]:
recovery_plot(sim.thresholds, rsm.thresholds, 'threshold', 'my_rsm_threshold_recovery.png')

Generate a table of person statistics (and check run time), and save to file

In [ ]:
%%time
rsm.person_stats_df(full=True)
rsm.person_stats.to_csv('rsm_person_stats.csv')

Check the person statistics table - view the first ten persons with `.head(10)`

In [ ]:
rsm.person_stats.head(10)

And the same recovery check for person locations, against `sim.persons`:

In [ ]:
recovery_plot(sim.persons, rsm.persons, 'person location', 'my_rsm_person_recovery.png')

Generate a table of test-level statistics (and check run time), and save to file

In [ ]:
%%time
rsm.test_stats_df()
rsm.test_stats.to_csv('rsm_test_stats.csv')

Check the test statistics table

In [ ]:
rsm.test_stats

Run a residual correlation analysis (and check run time), and save relevant output to file

In [ ]:
%%time
rsm.res_corr_analysis()
rsm.residual_correlations.to_csv('rsm_residual_correlations.csv')
rsm.loadings.to_csv('rsm_loadings.csv')

View the table of pairwise standard residual correlations

In [ ]:
round(rsm.residual_correlations, 3)

View the item loadings on the first principal component of the pairwise standard residual correlations (dimensionality test)

In [ ]:
round(rsm.loadings['PC 1'], 3)

Produce an item characteristic curve (item response function) for Item 2, with observed category means plotted and the central item location marked

In [ ]:
rsm.icc('Item_2', title='ICC for Item 2', obs=True, central_location=True, cat_highlight=3, xmin=-6, xmax=4, filename='my_rsm_icc')

Produce a set of category response curves for Item 2, with category 1 highlighted and the thresholds marked

In [ ]:
rsm.crcs('Item_2', thresh_lines=True, cat_highlight=1, obs=[2], xmin=-6, xmax=4, filename='my_rsm_crcs')

Produce a set of threshold characteristic curves for Item 2, with observed category means plotted for threshold 5, category 1 highlighted and the thresholds marked

In [ ]:
rsm.threshold_ccs('Item_2', thresh_lines=True, obs=[5], cat_highlight=1, xmin=-6, xmax=4, filename='my_rsm_threshold_ccs')

Produce an item information function curve for Item 2

In [ ]:
rsm.iic('Item_2', point_info_lines=[1], point_info_labels=True, title='Information for Item 2', xmin=-6, xmax=4, filename='my_rsm_iic')

Produce a test characteristic curve (test response function), with person locations corresponding to scores of 20 and 30 plotted.

In [ ]:
rsm.tcc(score_lines=[20, 30], score_labels=True, filename='my_rsm_tcc')

Produce a test information curve

In [ ]:
rsm.test_info(point_info_lines=[1], point_info_labels=True, filename='my_rsm_test_info_curve')

Produce a test CSEM (conditional standard error of measurement) curve, with the CSEM corresponding to a person location of -3 plotted

In [ ]:
rsm.test_csem(point_csem_lines=[-3], point_csem_labels=True, ymax=2.5, filename='my_rsm_csem_curve')

Produce a histogram of standardised residuals, with a normal distribution curve overlaid

In [ ]:
rsm.std_residuals_plot(bin_width=0.6, normal=True, filename='my_rsm_std_residuals_plot')